# Taxi Anomaly Detection via Zetaris Semantic Layer

This notebook runs the same IsolationForest anomaly detection as
`taxi_anomaly_detector.ipynb`, but sources data from the `taxi.taxi_trips`
unified view via the Zetaris Lightning REST API instead of reading parquet
files from MinIO.

In [ ]:
import os

ZETARIS_API_URL = os.environ.get("ZETARIS_API_URL", "")
ZETARIS_API_KEY = os.environ.get("ZETARIS_API_KEY", "")
ZETARIS_ORG_ID  = os.environ.get("ZETARIS_ORG_ID", "")

missing = []
if not ZETARIS_API_URL: missing.append("ZETARIS_API_URL")
if not ZETARIS_API_KEY: missing.append("ZETARIS_API_KEY")
if not ZETARIS_ORG_ID:  missing.append("ZETARIS_ORG_ID")
if missing:
    raise EnvironmentError(
        f"Missing required environment variable(s): {', '.join(missing)}. "
        "Set them in the workbench pod spec or export manually."
    )

masked_key = ZETARIS_API_KEY[:4] + "****" + ZETARIS_API_KEY[-4:]
print(f"Zetaris API URL : {ZETARIS_API_URL}")
print(f"Zetaris API Key : {masked_key}")
print(f"Zetaris Org ID  : {ZETARIS_ORG_ID}")

In [ ]:
import uuid

import pandas as pd
import requests

HEADERS = {
    "Authorization": f"Bearer {ZETARIS_API_KEY}",
    "Content-Type": "application/json",
    "X-Org-ID": ZETARIS_ORG_ID,
}

SQL = (
    "SELECT passenger_count, trip_distance, fare_amount, tip_amount, pickup_datetime "
    "FROM taxi.taxi_trips "
    "WHERE pickup_datetime >= '2025-10-01' AND pickup_datetime < '2025-11-01' "
    "LIMIT 10000"
)
PAGE_LIMIT = 100

query_token = None
all_records = []
try:
    resp = requests.post(
        f"{ZETARIS_API_URL}/api/v1.0/query/sql/start",
        headers={**HEADERS, "X-Request-ID": str(uuid.uuid4())},
        json={"select": SQL, "pageLimit": PAGE_LIMIT},
    )
    if resp.status_code == 401:
        raise RuntimeError("Authentication failed. Check ZETARIS_API_KEY.")
    resp.raise_for_status()

    data = resp.json()
    query_token = data["queryToken"]
    all_records.extend(data["records"])
    total_pages = data["totalPages"]

    for page_num in range(2, total_pages + 1):
        resp = requests.get(
            f"{ZETARIS_API_URL}/api/v1.0/query/sql/page",
            headers={**HEADERS, "X-Request-ID": str(uuid.uuid4())},
            params={"queryToken": query_token, "pageLimit": PAGE_LIMIT, "pageNumber": page_num},
        )
        resp.raise_for_status()
        all_records.extend(resp.json()["records"])

except requests.ConnectionError:
    raise RuntimeError("Cannot reach Zetaris API. Check ZETARIS_API_URL.")
finally:
    if query_token:
        requests.delete(
            f"{ZETARIS_API_URL}/api/v1.0/query/sql/close/{query_token}",
            headers={**HEADERS, "X-Request-ID": str(uuid.uuid4())},
        )

df = pd.DataFrame(all_records)
for col in ["passenger_count", "trip_distance", "fare_amount", "tip_amount"]:
    df[col] = pd.to_numeric(df[col])

print(f"Loaded {len(df):,} trips from taxi.taxi_trips via Zetaris API ({total_pages} pages)")
df.head()

In [ ]:
from sklearn.ensemble import IsolationForest

features = ["passenger_count", "trip_distance", "fare_amount", "tip_amount"]
df = df.dropna(subset=features)
X = df[features]

model = IsolationForest(contamination=0.02, random_state=42)
df["anomaly_score"] = model.fit_predict(X)
df["is_anomaly"] = df["anomaly_score"] == -1

anomalies = df[df["is_anomaly"]]
print(f"Detected {len(anomalies):,} anomalous trips ({len(anomalies) / len(df):.1%})")
anomalies.sort_values("fare_amount", ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
normal = df[~df["is_anomaly"]]
anomaly = df[df["is_anomaly"]]

ax.scatter(normal["trip_distance"], normal["fare_amount"], s=1, alpha=0.3, label="Normal")
ax.scatter(anomaly["trip_distance"], anomaly["fare_amount"], s=5, color="red", label="Anomaly")
ax.set_xlabel("Trip distance (miles)")
ax.set_ylabel("Fare amount ($)")
ax.set_title("Taxi fare anomalies")
ax.legend()
plt.show()